# 7.2 · MCP Apps：在对话里渲染交互界面

**MCP Apps: interactive UI in the conversation**

> **学完本课你能：** 使用 Python SDK 构建一个 MCP 应用（MCP App），将工具关联到带有 CSP 元数据的沙箱化 `ui://` HTML 资源，并讲清宿主↔应用之间的 postMessage 协议及其安全模型。

官方阅读材料：[MCP Apps overview](https://modelcontextprotocol.io/extensions/apps/overview) · [Build an MCP App](https://modelcontextprotocol.io/extensions/apps/build)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 使用 SDK 的 `Apps` 扩展构建 MCP 应用服务器
`mcp.server.apps.Apps` 是一个 `Extension`（扩展，Extension）。`@apps.tool(resource_uri=...)` 会在工具上打上 `_meta.ui`，`add_html_resource(...)` 则以 `text/html;profile=mcp-app` 提供一个 `ui://` 资源（Resource），并可附带 CSP 和权限。下面的 HTML 是一个原生 JS 应用，不借助任何 SDK，直接使用 postMessage 协议。注意 `content` 中的文本兜底。

In [ ]:
import json

from pydantic import BaseModel

from mcp import Client
from mcp.client.extension import advertise
from mcp.server import MCPServer
from mcp.server.apps import (APP_MIME_TYPE, EXTENSION_ID, Apps, ResourceCsp,
                             client_supports_apps)
from mcp.server.mcpserver import Context

SALES = {"Q3": {"EMEA": 120, "AMER": 200, "APAC": 90}}
DASHBOARD_URI = "ui://sales/dashboard.html"

DASHBOARD_HTML = """<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><title>Sales by region</title></head>
<body>
  <h1 id="title">Loading…</h1>
  <ul id="bars"></ul>
  <button id="refresh">Refresh</button>
  <script>
    let nextId = 1; const pending = {};
    const send = (method, params) => new Promise(resolve => {
      const id = nextId++; pending[id] = resolve;
      window.parent.postMessage({jsonrpc: "2.0", id, method, params}, "*");
    });
    const notify = (method, params) => window.parent.postMessage({jsonrpc: "2.0", method, params}, "*");
    const render = (sc) => {
      document.getElementById("title").textContent = "Sales " + sc.quarter;
      document.getElementById("bars").innerHTML = Object.entries(sc.regions)
        .map(([r, v]) => `<li>${r}: ${"#".repeat(v / 10)} ${v}</li>`).join("");
    };
    window.addEventListener("message", (e) => {
      const m = e.data;
      if (m.id && pending[m.id]) { pending[m.id](m.result); delete pending[m.id]; }
      if (m.method === "ui/notifications/tool-result") render(m.params.structuredContent);
    });
    send("ui/initialize", {appInfo: {name: "sales-dashboard", version: "1.0.0"},
                           appCapabilities: {}, protocolVersion: "2026-01-26"})
      .then(() => notify("ui/notifications/initialized", {}));
    document.getElementById("refresh").onclick = async () => {
      const r = await send("tools/call", {name: "refresh_sales", arguments: {quarter: "Q3"}});
      render(r.structuredContent);
      send("ui/update-model-context", {content: [{type: "text", text: "User refreshed the Q3 dashboard."}]});
    };
  </script>
</body>
</html>"""


class RegionSales(BaseModel):
    quarter: str
    regions: dict[str, int]


apps = Apps()


@apps.tool(resource_uri=DASHBOARD_URI, description="Show sales by region for a quarter (interactive).")
def sales_by_region(quarter: str, ctx: Context) -> RegionSales:
    data = RegionSales(quarter=quarter, regions=SALES[quarter])
    return data   # the SDK puts this in structuredContent AND serializes it as text content


@apps.tool(resource_uri=DASHBOARD_URI, visibility=["app"], description="Re-fetch the dashboard data.")
def refresh_sales(quarter: str) -> RegionSales:
    return RegionSales(quarter=quarter, regions=SALES[quarter])


@apps.tool(resource_uri=DASHBOARD_URI, visibility=["model"],
           description="Email the quarter's report to someone (model only: the UI must never trigger emails).")
def email_report(quarter: str, to: str) -> str:
    return f"Sent the {quarter} report to {to}."


apps.add_html_resource(DASHBOARD_URI, DASHBOARD_HTML, name="Sales dashboard",
                       description="Bar chart of sales per region",
                       csp=ResourceCsp(connect_domains=["https://api.example.com"]),
                       prefers_border=True)

server = MCPServer("sales", extensions=[apps])   # extensions are applied here: register app tools first


@server.tool()
def export_csv(quarter: str, ctx: Context) -> str:
    """Export a quarter's sales as CSV text (model-facing, no UI)."""
    note = "" if client_supports_apps(ctx) else "  # tip: an Apps-capable host would show a dashboard"
    return "region,sales\n" + "\n".join(f"{r},{v}" for r, v in SALES[quarter].items()) + note


print("extension id:", EXTENSION_ID, "| mime type:", APP_MIME_TYPE)

### 2. 发现：支持 Apps 的宿主看到了什么
宿主（Host）声明该扩展时带上<strong>必需的</strong> `mimeTypes` 设置。它从 `tools/list` 中读取 `_meta.ui`，然后用 `resources/read` 获取模板。

In [ ]:
HOST_APPS_CAP = advertise(EXTENSION_ID, {"mimeTypes": [APP_MIME_TYPE]})

async with Client(server, extensions=[HOST_APPS_CAP]) as host:
    print("server/discover extensions:", host.server_capabilities.extensions)
    tools = (await host.list_tools()).tools
    for t in tools:
        print(f"tool {t.name:<16} _meta={json.dumps(t.meta)}")
    res = await host.read_resource(DASHBOARD_URI)
    item = res.contents[0].model_dump(by_alias=True, exclude_none=True)
    print("\nresources/read ->", json.dumps({k: v for k, v in item.items() if k != "text"}, indent=2))
    print("HTML bytes:", len(item["text"]))

### 3. 宿主端校验：关联关系、MIME 类型与 CSP 头
一个谨慎的宿主（Host）在渲染任何东西之前都会检查每一个关联。它确认 `ui://` 协议、资源存在、MIME 类型（MIME type）完全匹配以及 HTML5 doctype。然后它按照规范的构造方式，<strong>只</strong>根据已声明的域名构建 iframe 的内容安全策略（Content Security Policy，CSP），并把沙箱化文档写入一个临时文件，将策略作为 `<meta>` 标签注入其中。

In [ ]:
import tempfile
from html.parser import HTMLParser
from pathlib import Path


def build_csp(ui_meta: dict | None) -> str:
    csp = (ui_meta or {}).get("csp")
    if csp is None:   # spec's restrictive default when ui.csp is omitted
        return ("default-src 'none'; script-src 'self' 'unsafe-inline'; style-src 'self' 'unsafe-inline'; "
                "img-src 'self' data:; media-src 'self' data:; connect-src 'none';")
    res = csp.get("resourceDomains", [])
    directives = [
        ["default-src", "'none'"],
        ["script-src", "'self'", "'unsafe-inline'", *res],
        ["style-src", "'self'", "'unsafe-inline'", *res],
        ["connect-src", "'self'", *csp.get("connectDomains", [])],
        ["img-src", "'self'", "data:", *res],
        ["font-src", "'self'", *res],
        ["media-src", "'self'", "data:", *res],
        ["frame-src", *(csp.get("frameDomains") or ["'none'"])],
        ["object-src", "'none'"],
        ["base-uri", *(csp.get("baseUriDomains") or ["'self'"])],
    ]
    return "; ".join(" ".join(d) for d in directives) + ";"


class ExternalRefs(HTMLParser):
    def __init__(self):
        super().__init__()
        self.refs = []

    def handle_starttag(self, tag, attrs):
        for name, value in attrs:
            if name in ("src", "href") and value and value.startswith(("http://", "https://")):
                self.refs.append((tag, value))


async def validate_ui_links(client) -> dict:
    report = {}
    for tool in (await client.list_tools()).tools:
        uri = ((tool.meta or {}).get("ui") or {}).get("resourceUri")
        if not uri:
            continue
        assert uri.startswith("ui://"), f"{tool.name}: resourceUri must use ui://"
        content = (await client.read_resource(uri)).contents[0]
        assert content.mime_type == APP_MIME_TYPE, f"{uri}: wrong mimeType {content.mime_type}"
        assert content.text.lstrip().lower().startswith("<!doctype html>"), f"{uri}: not an HTML5 document"
        parser = ExternalRefs()
        parser.feed(content.text)
        report[tool.name] = {"uri": uri, "ui_meta": (content.meta or {}).get("ui"),
                             "external_refs": parser.refs, "html": content.text}
    return report


async with Client(server, extensions=[HOST_APPS_CAP]) as host:
    report = await validate_ui_links(host)

entry = report["sales_by_region"]
csp_header = build_csp(entry["ui_meta"])
print("linked tools:", {name: r["uri"] for name, r in report.items()})
print("external refs in HTML (must be covered by resourceDomains):", entry["external_refs"])
print("\nCSP for the iframe:\n ", csp_header.replace("; ", ";\n  "))

sandbox_dir = Path(tempfile.mkdtemp())
page = entry["html"].replace("<head>", f'<head><meta http-equiv="Content-Security-Policy" content="{csp_header}">', 1)
(sandbox_dir / "dashboard.html").write_text(page)
print("\nwrote", sandbox_dir / "dashboard.html", f"({len(page)} bytes) - open it in a browser to see the static shell")
print("default CSP if ui.csp were omitted:", build_csp(None))

### 4. 可见性：模型的工具列表与应用的工具列表
执行可见性（Visibility）规则的是宿主（Host），而不是服务器。智能体（Agent）只能看到可见性包含 `"model"` 的工具，这也是缺省 `visibility` 时的默认值。应用只能调用可见性包含 `"app"` 的工具，同样是缺省时的默认值。`email_report` 关联了 UI，但其可见性是 `["model"]`，因此模型可以发送邮件，而仪表盘不行。

In [ ]:
def visibility(tool) -> list[str]:
    ui = (tool.meta or {}).get("ui") or {}
    return ui.get("visibility", ["model", "app"])   # spec default; a stricter host may refuse non-UI tools


async with Client(server, extensions=[HOST_APPS_CAP]) as host:
    all_tools = (await host.list_tools()).tools

model_tools = [t.name for t in all_tools if "model" in visibility(t)]
app_tools = [t.name for t in all_tools if "app" in visibility(t)]
print("tools the LLM is offered:   ", model_tools)
print("tools the app may call:     ", app_tools)
print("hidden from the LLM:        ", sorted(set(t.name for t in all_tools) - set(model_tools)))

### 5. 模拟宿主 ↔ 应用的 postMessage 流程
Python 跑不了 iframe，但这个协议就是普通的 JSON-RPC，所以我们可以同时扮演双方。`FakeView` 的行为与上面的 HTML 应用一致。`Host` 渲染它、完成握手、推送工具输入和结果，并在检查可见性（Visibility）之后把应用的 `tools/call` <strong>代理</strong>给真正的 MCP 服务器。每条消息都会连同方向一起打印出来。

In [ ]:
class FakeView:
    """Stands in for the iframe: same messages the JS app sends."""
    def __init__(self):
        self.next_id, self.state = 1, None

    def request(self, method, params):
        msg = {"jsonrpc": "2.0", "id": self.next_id, "method": method, "params": params}
        self.next_id += 1
        return msg

    def on_message(self, msg):
        if msg.get("method") == "ui/notifications/tool-result":
            self.state = msg["params"]["structuredContent"]
            print(f"      [view renders] {self.state}")


class Host:
    def __init__(self, client, tools):
        self.client, self.tools = client, {t.name: t for t in tools}
        self.initialized, self.model_context = False, None

    def to_view(self, view, msg):
        assert self.initialized or msg.get("id"), "Host MUST NOT message the view before initialized"
        print("HOST -> APP ", json.dumps(msg)[:150])
        view.on_message(msg)

    async def from_view(self, view, msg):
        print("APP  -> HOST", json.dumps(msg)[:150])
        method, params = msg["method"], msg.get("params", {})
        if method == "ui/initialize":
            reply = {"protocolVersion": "2026-01-26", "hostInfo": {"name": "notebook-host", "version": "0.1"},
                     "hostCapabilities": {"serverTools": {}, "openLinks": {}},
                     "hostContext": {"theme": "dark", "displayMode": "inline",
                                     "containerDimensions": {"width": 400, "maxHeight": 600}}}
        elif method == "ui/notifications/initialized":
            self.initialized = True
            return
        elif method == "tools/call":
            tool = self.tools.get(params["name"])
            if tool is None or "app" not in visibility(tool):
                return self.to_view(view, {"jsonrpc": "2.0", "id": msg["id"],
                                           "error": {"code": -32000, "message": "Tool not callable from app"}})
            result = await self.client.call_tool(params["name"], params.get("arguments"))
            reply = result.model_dump(by_alias=True, exclude_none=True, mode="json")
        elif method == "ui/update-model-context":
            self.model_context, reply = params, {}
        else:
            reply = {}
        self.to_view(view, {"jsonrpc": "2.0", "id": msg["id"], "result": reply})


async with Client(server, extensions=[HOST_APPS_CAP]) as client:
    host, view = Host(client, (await client.list_tools()).tools), FakeView()
    args = {"quarter": "Q3"}                                   # the model decided to call sales_by_region
    await host.from_view(view, view.request("ui/initialize", {
        "appInfo": {"name": "sales-dashboard", "version": "1.0.0"},
        "appCapabilities": {}, "protocolVersion": "2026-01-26"}))
    await host.from_view(view, {"jsonrpc": "2.0", "method": "ui/notifications/initialized", "params": {}})
    host.to_view(view, {"jsonrpc": "2.0", "method": "ui/notifications/tool-input", "params": {"arguments": args}})
    result = await client.call_tool("sales_by_region", args)
    host.to_view(view, {"jsonrpc": "2.0", "method": "ui/notifications/tool-result",
                        "params": result.model_dump(by_alias=True, exclude_none=True, mode="json")})
    print("\n-- user clicks Refresh --")
    await host.from_view(view, view.request("tools/call", {"name": "refresh_sales", "arguments": args}))
    await host.from_view(view, view.request("ui/update-model-context", {
        "content": [{"type": "text", "text": "User refreshed the Q3 dashboard."}]}))
    print("\n-- a compromised app tries a model-only tool --")
    await host.from_view(view, view.request("tools/call", {"name": "email_report",
                                                               "arguments": {**args, "to": "attacker@example.com"}}))
    print("\nmodel context stored by host:", host.model_context)

### 6. 优雅降级与护栏
不支持 Apps 的宿主（Host）仍然能拿到有用的文本：`content` 总是携带它，服务器还可以检查 `client_supports_apps`。这就是优雅降级（Graceful degradation）。SDK 还会在启动时捕获两种常见的错误配置：工具绑定到一个不存在的资源，以及 URI 未使用 `ui://` 协议。

In [ ]:
async with Client(server) as plain_host:     # no io.modelcontextprotocol/ui declared
    r = await plain_host.call_tool("sales_by_region", {"quarter": "Q3"})
    print("text a non-Apps host shows the model:", r.content[0].text.replace("\n", " "))
    print("export_csv in a non-Apps host:\n", (await plain_host.call_tool("export_csv", {"quarter": "Q3"})).content[0].text)

broken = Apps()


@broken.tool(resource_uri="ui://missing/app.html")
def orphan() -> str:
    return "never rendered"


try:
    MCPServer("broken", extensions=[broken])
except ValueError as exc:
    print("\nSDK refused the server:", exc)
try:
    broken.add_html_resource("https://evil.example/app.html", "<!DOCTYPE html>")
except ValueError as exc:
    print("SDK refused the resource:", exc)

### 动手练习
1. 在仪表盘的 `ResourceCsp` 中加入 `resource_domains=["https://cdn.jsdelivr.net"]`，并在 HTML 中加入一个 `<script src="https://cdn.jsdelivr.net/npm/chart.js">` 标签。重新运行第 3 节，确认该外部引用现在已被 `script-src` 覆盖。然后删除这条 CSP 条目，编写一个检查来标记未被覆盖的引用。
2. 扩展 `Host.from_view`：处理 `ui/message` 时把消息追加到一个 `chat` 列表中，处理 `ui/open-link` 时拒绝任何非 `https` 的 URL。
3. 当 `client_supports_apps(ctx)` 为真时，让 `sales_by_region` 返回更短的文本，因为 UI 会展示细节。为了让模型能对结果进行推理，这段文本仍然必须包含什么？
4. 模拟拆除流程：让宿主发送 `ui/resource-teardown` <strong>请求</strong>，并在“销毁”视图之前等待视图的响应。